# Project Approval

## 1. Dataset

See [Data Card](../docs/data_card.md) for dataset information, structure,
licensing, and limitations.

## 2. Initial Dataset Check

### Class Distribution

In [8]:
# Check and confirm the number of images in each class for both training and testing datasets

from pathlib import Path

DATA_DIR = Path("../data")

for split in ["Training", "Testing"]:
    split_dir = DATA_DIR / split # joins path together, not divide (ex. DATA_DIR / "Training" = "../data/Training")

    counts = {}

    for class_dir in sorted(split_dir.iterdir()): # Loop through every item in training or test folder
        if class_dir.is_dir():
            counts[class_dir.name] = len([
                p for p in class_dir.iterdir()
                if p.is_file()
            ])

    total = sum(counts.values())

    print(f"\n{split}")
    print("-" * 30)

    for class_name, count in counts.items():
        percentage = count / total * 100
        print(f"{class_name}: {count} ({percentage:.2f}%)")


Training
------------------------------
glioma: 1400 (25.00%)
meningioma: 1400 (25.00%)
notumor: 1400 (25.00%)
pituitary: 1400 (25.00%)

Testing
------------------------------
glioma: 400 (25.00%)
meningioma: 400 (25.00%)
notumor: 400 (25.00%)
pituitary: 400 (25.00%)


### File Formats

In [ ]:
# Count the number of files for each file extension in the data directory

from collections import Counter

extensions = Counter()

for path in DATA_DIR.rglob("*"): # Recursively search for all files in the data directory and its subdirectories
    if path.is_file():
        extensions[path.suffix.lower()] += 1

print(extensions)

# Ignore ".md: 1" as it's the README file created for the public audience.

Counter({'.jpg': 7200, '.md': 1})


### Corrupt Files

In [ ]:
# Check for corrupt image files in the data directory

from PIL import Image

corrupt_files = []

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            corrupt_files.append(path)

print("Corrupt files:", len(corrupt_files))

for path in corrupt_files:
    print(path)

# Ignore readme output

Corrupt files: 1
..\data\README.md


### Image Dimensions and Colour Modes

In [ ]:
# Check the dimensions and colour modes of the images in the data directory

dimensions = Counter()
modes = Counter()

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        try:
            with Image.open(path) as img:
                dimensions[img.size] += 1
                modes[img.mode] += 1
        except Exception:
            pass

print("Most common dimensions:")
for size, count in dimensions.most_common(10):
    print(size, count)

print("\nColour modes:")
for mode, count in modes.items():
    print(mode, count)

# Colour Mode explanation: L = grayscale, RGB = color, RGBA = color with alpha channel (transparency), P = palette

Most common dimensions:
(512, 512) 5014
(225, 225) 338
(630, 630) 90
(201, 251) 57
(228, 221) 51
(232, 217) 50
(442, 442) 48
(236, 236) 48
(150, 198) 44
(200, 252) 43

Colour modes:
L 3067
RGB 4129
RGBA 3
P 1


### Exact Duplicate Files

Variable dimensions and colour modes were found, meaning preprocessing will be necessary.

In [ ]:
# Check for exact duplicate files in the data directory 
# Reads an entire file's bytes, if two files have the same bytes, they are considered duplicates (reliable method)
# Keep in mind it's checking if the two files are byte-for-byte identical, not if they are visually identical

import hashlib
from collections import defaultdict

hashes = defaultdict(list)

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        file_hash = hashlib.md5(path.read_bytes()).hexdigest() # Calculate the MD5 hash of the file's contents
        hashes[file_hash].append(path)

duplicates = { # Create a dictionary of duplicate files, where the key is the file hash and the value is a list of paths to the duplicate files
    file_hash: paths
    for file_hash, paths in hashes.items()
    if len(paths) > 1
}

print("Duplicate groups:", len(duplicates))

for paths in list(duplicates.values())[:10]:
    print("\n".join(str(path) for path in paths))
    print()

Duplicate groups: 153
..\data\Training\meningioma\Tr-aug-me_89.jpg
..\data\Training\meningioma\Tr-me_889.jpg

..\data\Training\meningioma\Tr-aug-me_90.jpg
..\data\Training\meningioma\Tr-me_1242.jpg

..\data\Training\meningioma\Tr-aug-me_91.jpg
..\data\Training\meningioma\Tr-me_245.jpg

..\data\Training\meningioma\Tr-aug-me_92.jpg
..\data\Training\meningioma\Tr-me_268.jpg

..\data\Training\meningioma\Tr-aug-me_94.jpg
..\data\Training\meningioma\Tr-me_824.jpg

..\data\Training\meningioma\Tr-aug-me_95.jpg
..\data\Training\meningioma\Tr-me_1295.jpg

..\data\Training\meningioma\Tr-aug-me_97.jpg
..\data\Training\meningioma\Tr-me_532.jpg

..\data\Training\meningioma\Tr-aug-me_98.jpg
..\data\Training\meningioma\Tr-me_1274.jpg

..\data\Training\meningioma\Tr-me_1094.jpg
..\data\Training\meningioma\Tr-me_219.jpg

..\data\Training\meningioma\Tr-me_1101.jpg
..\data\Training\meningioma\Tr-me_836.jpg



## 3. Initial EDA

See [EDA notebook](./eda.ipynb).

## 4. Target and Task

The primary task here is that given a 2D brain MRI image, the model will predict one of 4 classes:

- Glioma
- Meningioma
- Pituitary tumour
- No tumour

Three machine learning approaches will be compared under a consistent preprocessing and evaluation framework:

- KNN
- Decision Tree
- CNN

### Objective/Task

How do KNN, Decision Tree, and CNN models compare in classifying brain MRI images into glioma, meningioma, pituitary tumour, and no tumour?

## 5. Risks

## 6. Train / Validation / Test Strategy

## 7. Primary Metric

## 8. Feasibility and Compute Plan